# 🎙️ HireVoice AI — Interactive Flow Tester (`notebook.ipynb`)
Use this notebook to test every part of your hackathon project independently:
1. **Environment & API Key Check**
2. **Flow 1: Credit-Free Text Mode (`use_voice=False`)** — Test vague role clarification, JD generation, AI field suggestions, and Typeform creation.
3. **Flow 2: Direct Typeform Create API Test** — Verify your `TYPEFORM_API_KEY` generates a live shareable form link.
4. **Flow 3: Sarvam AI Voice Test (`use_voice=True`)** — Test `bulbul:v3` TTS audio playback and `saaras:v3` STT transcription right inside Jupyter.

In [ ]:
# Cell 1: Load Environment Variables & Import the LangGraph Agent
import os
from dotenv import load_dotenv
from IPython.display import Markdown, Audio, display
from langchain_core.messages import HumanMessage

load_dotenv()

print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("TYPEFORM_API_KEY configured:", bool(os.getenv("TYPEFORM_API_KEY")))
print("SARVAM_API_KEY configured:", bool(os.getenv("SARVAM_API_KEY")))

from main import hiring_graph, create_typeform_form, sarvam_text_to_speech, sarvam_speech_to_text

## Flow 1: Multi-Turn LangGraph Conversation (`use_voice = False` — 0 Sarvam Credits)
Run the helper function below to simulate the entire 4-turn conversation:
- **Turn 1:** User gives a vague prompt (*"I need to hire a developer"*) -> Agent asks clarifying questions.
- **Turn 2:** User provides specifics (*"Senior Backend Engineer in Go and Kubernetes, 4+ years experience, Bengaluru hybrid"*) -> Agent drafts JD & recommends role-specific fields.
- **Turn 3:** User modifies fields (*"Add your recommended fields and also ask for current notice period"*).
- **Turn 4:** User approves (*"Okay I am satisfied with these inputs, create the form"*) -> Agent triggers `form_builder_node` and outputs the live Typeform link.

In [ ]:
# Cell 2: Helper Function to Run a Turn in Text Mode (Zero Voice Credits)
 THREAD_ID = "notebook-test-thread-1"
config = {"configurable": {"thread_id": THREAD_ID}}

def run_agent_turn(user_text: str):
    print(f"\n🧑 USER: {user_text}")
    state = hiring_graph.invoke({"messages": [HumanMessage(content=user_text)]}, config=config)
    print(f"🤖 AGENT (Voice Script): {state['spoken_response']}")
    print(f"📍 STAGE: {state['stage']} | MISSING INFO: {state.get('missing_info')}")
    print(f"📋 EXTRACTED SLOTS: {state.get('jd_slots')}")
    if state.get('recommended_fields'):
        print(f"✨ AI RECOMMENDED FIELDS: {[f['label'] for f in state['recommended_fields']]}")
    if state.get('form_fields'):
        print(f"✅ APPROVED FORM FIELDS: {[f['label'] for f in state['form_fields']]}")
    if state.get('form_url'):
        print(f"🔗 LIVE TYPEFORM LINK: {state['form_url']}")
    return state

In [ ]:
# Cell 3: Turn 1 - Vague input triggers clarification questions
state = run_agent_turn("Hey, I want to hire an engineer for my team.")

In [ ]:
# Cell 4: Turn 2 - Provide specific role details -> Generates JD & recommends fields
state = run_agent_turn("We need a Senior AI Backend Engineer with 3 to 5 years of experience, based in Bengaluru Hybrid. Must know Python, LangGraph, FastAPI, and PostgreSQL.")
if state.get('draft_jd'):
    display(Markdown("### 📄 Generated Job Description Preview\n" + state['draft_jd']))

In [ ]:
# Cell 5: Turn 3 - Accept AI field suggestions and add a custom field
state = run_agent_turn("Yes, add all your recommended fields, and also add a number field asking for their Notice Period in days.")

In [ ]:
# Cell 6: Turn 4 - Final approval triggers Typeform generation
state = run_agent_turn("Okay I am satisfied with these inputs, please proceed and create the form!")

## Flow 2: Standalone Typeform Create API Test
Run this cell to directly test creating a Typeform with various field types (`short_text`, `email`, `website`, `number`, `multiple_choice`).

In [ ]:
# Cell 7: Direct Typeform API Smoke Test
sample_fields = [
    {"label": "Candidate Full Name", "field_type": "short_text", "required": True},
    {"label": "Email Address", "field_type": "email", "required": True},
    {"label": "GitHub or Portfolio Link", "field_type": "website", "required": True},
    {"label": "Years of Production Python Experience", "field_type": "number", "required": True},
    {"label": "Preferred Work Mode", "field_type": "multiple_choice", "required": True, "options": ["Bengaluru Hybrid", "Remote", "Relocate to Bengaluru"]}
]

url = create_typeform_form(
    role_title="Senior AI Engineer (Hackathon Demo)",
    jd_markdown="# Senior AI Engineer\nBuild multi-agent systems.",
    fields=sample_fields
)
print("Created Typeform URL:", url)

## Flow 3: Sarvam AI Voice Test (`TTS` + `STT` Round-Trip)
Run this cell when you're ready to test your `SARVAM_API_KEY`. It synthesizes speech using **Bulbul (`bulbul:v3`)**, plays the audio right inside the notebook, and sends the generated WAV bytes back into **Saaras (`saaras:v3`)** to verify Speech-to-Text transcription!

In [ ]:
# Cell 8: Sarvam TTS -> Audio Playback -> Sarvam STT Round-Trip Test
import base64

sample_voice_line = "Hi! I have drafted the Senior Backend Engineer JD. Should I add GitHub profile and notice period to your Typeform?"
b64_audio = sarvam_text_to_speech(sample_voice_line)

if b64_audio:
    wav_bytes = base64.b64decode(b64_audio)
    print("✅ Sarvam TTS succeeded! Playing audio below:")
    display(Audio(wav_bytes, autoplay=False))
    
    # Feed the synthesized WAV directly into Sarvam STT to verify transcription without a mic!
    transcript = sarvam_speech_to_text(wav_bytes, filename="test_tts.wav")
    print("🎙️ Sarvam STT Round-Trip Transcript:", transcript)
else:
    print("ℹ️ Set a valid SARVAM_API_KEY in .env to test voice synthesis and transcription.")